[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ThanhDatVN/AI_journey/blob/main/notebooks/30_sequence_models/01_rnn_from_scratch.ipynb)

# Bài 01 — Mạng nơ-ron hồi quy (RNN) from scratch

**🎯 Mục tiêu**
- Dữ liệu chuỗi và các dạng bài toán (one-to-many, many-to-one, many-to-many).
- Ô RNN: trạng thái ẩn mang "trí nhớ", **chia sẻ trọng số theo thời gian**; trải (unroll) theo thời gian.
- **Backpropagation through time (BPTT)** from scratch bằng NumPy; kiểm tra gradient.
- Mô hình ngôn ngữ **mức ký tự** trên review tiếng Việt: huấn luyện, sinh văn bản, nhiệt độ lấy mẫu; đối chiếu `nn.RNN`.

In [ ]:
# ⚙️ Setup — chạy được cả local lẫn Google Colab (local: đã `pip install -e .` thì bỏ qua)
import sys
if "google.colab" in sys.modules:
    !test -d /content/ml-course || git clone -q --depth 1 https://github.com/ThanhDatVN/AI_journey /content/ml-course
    !pip install -q -e /content/ml-course
    sys.path.insert(0, "/content/ml-course/src")
    %cd /content/ml-course/notebooks/30_sequence_models

In [ ]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from ml_course.visualization import set_style

try:
    import torch
    from torch import nn
except ImportError as e:
    raise ImportError("Chương này cần PyTorch: pip install -e \".[dl]\"") from e

set_style()

## 1. Dữ liệu chuỗi

| Dạng | Ví dụ |
|---|---|
| many-to-one | phân loại cảm xúc một câu; dự đoán lớp của chuỗi cảm biến |
| one-to-many | sinh mô tả cho một ảnh |
| many-to-many (đồng bộ) | gán nhãn từ loại từng từ; dự báo từng bước |
| many-to-many (lệch) | dịch máy, tóm tắt (encoder–decoder — bài 04) |

Đặc điểm: độ dài **thay đổi**, **thứ tự** quan trọng ("không tốt" ≠ "tốt, không..."), phụ thuộc có thể **xa**. MLP cần đầu vào cố định; CNN 1D chỉ nhìn cửa sổ hữu hạn. RNN đọc từng phần tử và cập nhật một **trạng thái ẩn** tóm tắt quá khứ.

## 2. Ô RNN

$$\mathbf h_t = \tanh(W_{xh}\mathbf x_t + W_{hh}\mathbf h_{t-1} + \mathbf b_h), \qquad \mathbf y_t = W_{hy}\mathbf h_t + \mathbf b_y$$

Cùng $W_{xh}, W_{hh}, W_{hy}$ dùng ở **mọi** bước thời gian (như CNN chia sẻ trọng số theo không gian). Trải theo thời gian, RNN là một mạng rất sâu (độ sâu = độ dài chuỗi) với các lớp dùng chung trọng số.

```
x₁ → [h₁] → y₁
       ↓
x₂ → [h₂] → y₂
       ↓
x₃ → [h₃] → y₃   ...
```

## 3. Bài toán: mô hình ngôn ngữ mức ký tự

Đọc từng ký tự, dự đoán ký tự **tiếp theo**. Ngữ liệu: 3 000 review sản phẩm tiếng Việt (mỗi review một dòng).

In [ ]:
reviews = pd.read_csv("../../data/sample/reviews_vi.csv")["text"].tolist()
corpus = "\n".join(reviews) + "\n"
chars = sorted(set(corpus))
V = len(chars)
char_to_ix = {c: i for i, c in enumerate(chars)}
ix_to_char = {i: c for c, i in char_to_ix.items()}
data = np.array([char_to_ix[c] for c in corpus])
print(f"{len(corpus):,} ký tự | {V} ký tự khác nhau (có dấu tiếng Việt)")
print(corpus[:160])

## 4. Forward và BPTT

Loss = tổng cross-entropy dự đoán ký tự tiếp theo ở mọi bước. **BPTT**: chạy forward qua $T$ bước (lưu mọi $\mathbf h_t$), rồi lan truyền ngược từ $t = T$ về $1$. Gradient theo $\mathbf h_t$ nhận hai phần: từ đầu ra tại $t$ **và** từ $\mathbf h_{t+1}$:
$$\delta\mathbf h_t = W_{hy}^\top(\hat{\mathbf p}_t - \mathbf e_{y_t}) + W_{hh}^\top\big[(1 - \mathbf h_{t+1}^2)\odot\delta\mathbf h_{t+1}\big]$$
Gradient của trọng số dùng chung = **tổng** qua mọi bước thời gian.

In [ ]:
def rnn_loss_and_grads(inputs, targets, h_prev, params):
    """inputs, targets: list chỉ số ký tự; h_prev: (H, 1). Trả về loss, grads, h cuối."""
    Wxh, Whh, Why, bh, by = params
    xs, hs, ps = {}, {-1: h_prev}, {}
    loss = 0.0
    for t, ix in enumerate(inputs):                                     # forward
        xs[t] = np.zeros((V, 1))
        xs[t][ix] = 1
        hs[t] = np.tanh(Wxh @ xs[t] + Whh @ hs[t - 1] + bh)
        logits = Why @ hs[t] + by
        logits -= logits.max()
        ps[t] = np.exp(logits) / np.exp(logits).sum()
        loss -= np.log(ps[t][targets[t], 0])
    dWxh, dWhh, dWhy = np.zeros_like(Wxh), np.zeros_like(Whh), np.zeros_like(Why)
    dbh, dby = np.zeros_like(bh), np.zeros_like(by)
    dh_next = np.zeros_like(h_prev)
    for t in reversed(range(len(inputs))):                              # backward through time
        dy = ps[t].copy()
        dy[targets[t]] -= 1                                             # p̂ − one-hot
        dWhy += dy @ hs[t].T
        dby += dy
        dh = Why.T @ dy + dh_next                                       # từ đầu ra + từ bước sau
        draw = (1 - hs[t] ** 2) * dh                                    # qua tanh
        dWxh += draw @ xs[t].T
        dWhh += draw @ hs[t - 1].T
        dbh += draw
        dh_next = Whh.T @ draw
    return loss, (dWxh, dWhh, dWhy, dbh, dby), hs[len(inputs) - 1]


def init_params(H, seed=0):
    r = np.random.default_rng(seed)
    return [r.normal(0, 0.01, (H, V)), r.normal(0, 0.01, (H, H)), r.normal(0, 0.01, (V, H)), np.zeros((H, 1)), np.zeros((V, 1))]

### Kiểm tra gradient

In [ ]:
params_chk = init_params(8, seed=1)
for p in params_chk[:3]:
    p *= 50                                                             # trọng số lớn hơn để gradient không quá nhỏ
inp, tgt = list(data[:6]), list(data[1:7])
h0 = np.zeros((8, 1))
_, grads, _ = rnn_loss_and_grads(inp, tgt, h0, params_chk)
rng = np.random.default_rng(0)
worst = 0.0
for p, g in zip(params_chk, grads):
    for _ in range(5):
        idx = tuple(rng.integers(0, s) for s in p.shape)
        old = p[idx]
        p[idx] = old + 1e-5
        lp = rnn_loss_and_grads(inp, tgt, h0, params_chk)[0]
        p[idx] = old - 1e-5
        lm = rnn_loss_and_grads(inp, tgt, h0, params_chk)[0]
        p[idx] = old
        num = (lp - lm) / 2e-5
        worst = max(worst, abs(num - g[idx]) / max(abs(num) + abs(g[idx]), 1e-10))
print(f"sai lệch tương đối lớn nhất (25 tham số ngẫu nhiên): {worst:.2e}")

## 5. Huấn luyện và sinh văn bản

Cắt ngữ liệu thành các đoạn 25 ký tự; giữ trạng thái ẩn nối tiếp giữa các đoạn nhưng **cắt gradient** tại biên đoạn (**truncated BPTT**). Optimizer Adagrad và **cắt gradient** về $[-5, 5]$ (bài 02 giải thích vì sao).

In [ ]:
def sample(params, h, seed_ix, n, temperature=1.0, seed=0):
    Wxh, Whh, Why, bh, by = params
    r = np.random.default_rng(seed)
    x = np.zeros((V, 1))
    x[seed_ix] = 1
    out = []
    for _ in range(n):
        h = np.tanh(Wxh @ x + Whh @ h + bh)
        logits = (Why @ h + by).ravel() / temperature
        p = np.exp(logits - logits.max())
        p /= p.sum()
        ix = r.choice(V, p=p)
        x = np.zeros((V, 1))
        x[ix] = 1
        out.append(ix_to_char[ix])
    return "".join(out)


H, seq_len, lr = 100, 25, 0.1
params = init_params(H)
memory = [np.zeros_like(p) for p in params]                            # Adagrad
smooth_loss = -np.log(1.0 / V) * seq_len
start_ix = char_to_ix["\n"]
print("TRƯỚC khi train:", repr(sample(params, np.zeros((H, 1)), start_ix, 80)))
history, pos, h_prev = [], 0, np.zeros((H, 1))
t0 = time.perf_counter()
for it in range(6000):
    if pos + seq_len + 1 >= len(data):
        pos, h_prev = 0, np.zeros((H, 1))
    inputs, targets = list(data[pos:pos + seq_len]), list(data[pos + 1:pos + seq_len + 1])
    loss, grads, h_prev = rnn_loss_and_grads(inputs, targets, h_prev, params)
    for p, g, m in zip(params, grads, memory):
        np.clip(g, -5, 5, out=g)
        m += g * g
        p -= lr * g / np.sqrt(m + 1e-8)
    smooth_loss = 0.999 * smooth_loss + 0.001 * loss
    history.append(smooth_loss / seq_len)
    pos += seq_len
print(f"6 000 bước trong {time.perf_counter() - t0:.0f}s | loss/ký tự: {-np.log(1.0 / V):.2f} (đoán đều) → {history[-1]:.2f}")

In [ ]:
plt.plot(history)
plt.xlabel("bước")
plt.ylabel("cross-entropy / ký tự (làm trơn)")
plt.show()
for temp in [0.5, 1.0, 1.5]:
    print(f"--- nhiệt độ {temp} ---")
    print(sample(params, np.zeros((H, 1)), start_ix, 240, temperature=temp, seed=1))

Chỉ với vài nghìn bước, RNN học được: chính tả tiếng Việt có dấu, cụm từ thường gặp ("giao hàng", "sản phẩm"), dấu phẩy, xuống dòng kết thúc review. **Nhiệt độ** điều chỉnh độ "liều": thấp → an toàn, lặp lại cụm phổ biến; cao → đa dạng nhưng nhiều lỗi. (Mô hình ngôn ngữ lớn — chương 31 — dùng đúng nguyên lý "dự đoán token tiếp theo", với Transformer thay cho RNN.)

## 6. Đối chiếu `nn.RNN`

`nn.RNN` tính $\mathbf h_t = \tanh(W_{ih}\mathbf x_t + \mathbf b_{ih} + W_{hh}\mathbf h_{t-1} + \mathbf b_{hh})$ — gán trọng số của ta vào và so sánh:

In [ ]:
Wxh, Whh, Why, bh, by = params
rnn = nn.RNN(V, H, batch_first=True)
with torch.no_grad():
    rnn.weight_ih_l0.copy_(torch.tensor(Wxh))
    rnn.weight_hh_l0.copy_(torch.tensor(Whh))
    rnn.bias_ih_l0.copy_(torch.tensor(bh.ravel()))
    rnn.bias_hh_l0.zero_()
seq = data[:30]
x_onehot = torch.zeros(1, len(seq), V)
x_onehot[0, np.arange(len(seq)), seq] = 1
out_torch, _ = rnn(x_onehot)
h = np.zeros((H, 1))
hs_np = []
for ix in seq:
    x = np.zeros((V, 1))
    x[ix] = 1
    h = np.tanh(Wxh @ x + Whh @ h + bh)
    hs_np.append(h.ravel())
print(f"sai lệch lớn nhất giữa trạng thái ẩn NumPy và nn.RNN: {np.abs(np.array(hs_np) - out_torch[0].detach().numpy()).max():.2e}")

Trong thực tế dùng `nn.Embedding` thay cho one-hot (tra bảng vector — tương đương nhân one-hot với ma trận), `batch_first=True`, và xử lý batch nhiều chuỗi song song.

## 🧠 Kiểm tra nhanh

1. Trạng thái ẩn của RNN đóng vai trò gì? Vì sao chia sẻ trọng số theo thời gian?
2. Viết công thức cập nhật $\mathbf h_t$. Số tham số có phụ thuộc độ dài chuỗi không?
3. Trong BPTT, $\delta\mathbf h_t$ nhận gradient từ đâu?
4. Truncated BPTT là gì? Vì sao cần?
5. Nhiệt độ khi lấy mẫu ảnh hưởng thế nào?

## 📌 Tóm tắt

- RNN đọc chuỗi từng bước, cập nhật trạng thái ẩn với trọng số dùng chung; trải ra là mạng sâu theo thời gian.
- BPTT: forward lưu mọi trạng thái, backward cộng gradient qua thời gian; truncated BPTT cho chuỗi dài; luôn kiểm tra gradient.
- Mô hình ngôn ngữ ký tự: dự đoán ký tự tiếp theo, sinh văn bản với nhiệt độ.

## 📚 Tài liệu tham khảo

- Elman (1990) — *Finding Structure in Time*
- Karpathy (2015) — *The Unreasonable Effectiveness of Recurrent Neural Networks* và *min-char-rnn.py*
- Goodfellow, Bengio & Courville (2016) — *Deep Learning*, ch. 10